In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':', 1)
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
# metadata_df = metadata_df.reset_index()

print(metadata_df.shape)
metadata_df.head()


(28, 18)


,sample_title,patient id,anti-pd-1 response,study site,gender,age (yrs),disease status,overall survival (days),vital status,previous mapki,anatomical location,nras,treatment,biopsy time,tissue,stranded/unstranded rnaseq,nf1,braf
samples,,,,,,,,,,,,,,,,,,
GSM2069823,Pt1,Pt1,Progressive Disease,UCLA,F,66,M1b,607,Dead,N,"L post. Thigh, inf., SC",Q61L,Pembrolizumab,pre-treatment,Melanoma biopsies,unstranded,NaN,NaN
GSM2069824,Pt2,Pt2,Partial Response,UCLA,M,55,M1c,927,Alive,N,Lung,NaN,Pembrolizumab,pre-treatment,Melanoma biopsies,unstranded,"Frame_Shift_Ins H31fs, Splice_Site",NaN
GSM2069825,Pt4,Pt4,Partial Response,UCLA,M,62,M1c,948,Alive,Y,Adrenal,NaN,Pembrolizumab,pre-treatment,Melanoma biopsies,unstranded,S2597*,L331F
GSM2069826,Pt5,Pt5,Partial Response,UCLA,M,61,M1c,439,Alive,N,"L Chestwall, SC",NaN,Pembrolizumab,pre-treatment,Melanoma biopsies,unstranded,NaN,V600E
GSM2069827,Pt6,Pt6,Partial Response,UCLA,M,51,M1c,882,Alive,Y,"L forearm, SC",NaN,Pembrolizumab,pre-treatment,Melanoma biopsies,unstranded,NaN,V600E


In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"


df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None


url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)


df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined.reset_index()
df_combined.to_csv(data_file_name, index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_11122/779919605.py:12: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,patient id,anti-pd-1 response,study site,gender,age (yrs),disease status,overall survival (days),vital status,previous mapki,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM2069823,Pt1,Pt1,Progressive Disease,UCLA,F,66,M1b,607,Dead,N,...,183692,79,94,90,106449,35525,3499,107038,127,2120
GSM2069824,Pt2,Pt2,Partial Response,UCLA,M,55,M1c,927,Alive,N,...,404290,161,188,198,153689,66501,4124,233872,140,2023
GSM2069825,Pt4,Pt4,Partial Response,UCLA,M,62,M1c,948,Alive,Y,...,1501477,1109,1470,1747,681687,241629,8809,906518,688,8849
GSM2069826,Pt5,Pt5,Partial Response,UCLA,M,61,M1c,439,Alive,N,...,13076,4,4,5,6674,2166,221,8156,8,158
GSM2069827,Pt6,Pt6,Partial Response,UCLA,M,51,M1c,882,Alive,Y,...,2358153,2670,3887,4698,1798184,599466,15513,1508670,1367,12451
GSM2069828,Pt7,Pt7,Progressive Disease,UCLA,F,55,M1c,662,Dead,N,...,851017,2768,3600,3844,539332,232311,14444,770608,743,7250
GSM2069829,Pt8,Pt8,Complete Response,UCLA,M,69,M1a,NA,Alive,N,...,513807,348,454,507,127117,49003,3268,351087,387,1619
GSM2069830,Pt9,Pt9,Complete Response,UCLA,M,68,M0,1054,Alive,N,...,821701,641,982,1053,229239,171416,16395,500930,431,7987
GSM2069831,Pt10,Pt10,Progressive Disease,UCLA,M,60,M1a,387,Alive,Y,...,897765,495,583,525,417481,125901,8754,465938,354,9453


In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)

desc_data['data_summary'] = {
    'samples': df_combined.shape[0],
    'clinical_features': metadata_df.shape[1],
    'genes': df_raw_counts.shape[1]
}
print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated
